# Auditoria da malha rodoviaria
Dados reais do cadastro DER/SP, Censo 2022 e IBGE 2019.
Processamento completo: ../scripts/auditar_e_construir_dashboard.py.
Esta verificacao registra as conciliacoes e evidencia os conflitos do legado.

In [1]:
from pathlib import Path
import json, gzip
root = Path.cwd()
if not (root / 'relatorio').exists(): root = root.parent
data = json.loads((root/'relatorio/assets/data/indicadores.json').read_text(encoding='utf-8'))
audit = json.loads((root/'relatorio/assets/data/auditoria.json').read_text(encoding='utf-8'))
print(audit['legado'])

{'extensao_municipal_km': 21587.165999999997, 'populacao_rotulada_2022': 46024937, 'populacao_censo2022_confirmada': 44411238, 'concessionada_km': 9245.51, 'grafico_concessao_publicado_km': [6847, 15267], 'histograma_publicado': [285, 168, 92, 48, 27, 25], 'histograma_recalculado': [267, 189, 89, 40, 35, 25], 'densidade_pop_escala_real': 1000, 'densidade_pop_rotulo': 10000, 'extensao_ra_geometrica_km': 21681.03}


In [2]:
roads=data['trechos']
for level,units in data['recortes'].items():
    extent=sum(sum(r['extensao'] for r in roads if r[level]==u['id']) for u in units)
    population=sum(u['populacao'] for u in units)
    area=sum(u['area'] for u in units)
    assert abs(extent-21682.703)<1e-6 and population==44411238
    print(level,len(units),round(extent,3),population,round(area,2))

municipios 645 21682.703 44411238 248219.48
imediatas 53 21682.703 44411238 248219.48
intermediarias 11 21682.703 44411238 248219.48
administrativas 16 21682.703 44411238 248219.48
zee 9 21682.703 44411238 248219.48


In [3]:
print(audit['urbanizacao'])
print(audit['associacao_territorial'])
assert all(0<=r['proporcao_urbanizada']<=1 for r in roads)
assert len({r['id'] for r in roads})==len(roads)
print('Unique road IDs and urban proportions: PASS')

{'fonte': 'Áreas Urbanizadas do Brasil 2019, IBGE', 'sha256_shp': 'fc52d4b02fd14ca14ef4cf9ecd0f6ef03bdd8bb704c775165d090aca2d265ae0', 'poligonos_bbox_tipo_area_urbanizada': 18846, 'criterio': 'Tipo = Área urbanizada; exclui loteamentos vazios, equipamentos e vazios intraurbanos', 'extensao_urbanizada_estimada_km': 3026.704967676785, 'extensao_por_classe_km': {'Urbano': 2286.042, 'Misto': 3987.712, 'Rural': 15408.949}}
{'metodo': 'Maior área de interseção municipal com RA/ZEE; RGI/RGINT por código IBGE', 'min_proporcao': 0.92303, 'abaixo_99_porcento': [{'municipio': 'Guarujá', 'codigo': '3518701', 'recorte': 'zee', 'proporcao_area': 0.986257}, {'municipio': 'Rubinéia', 'codigo': '3544509', 'recorte': 'zee', 'proporcao_area': 0.92303}, {'municipio': 'Panorama', 'codigo': '3535408', 'recorte': 'zee', 'proporcao_area': 0.984134}, {'municipio': 'Rosana', 'codigo': '3544251', 'recorte': 'zee', 'proporcao_area': 0.970393}, {'municipio': 'São Bento do Sapucaí', 'codigo': '3548609', 'recorte': 